# Stage 2: Data Preprocessing & Target Dataset Construction

**Project**: EDA-Driven Behavioral Feature Engineering for Human–AI Chess Move Detection  
**Goal**: Execute the complete preprocessing pipeline to convert raw PGN games into the clean, verified move-level dataset `data/processed/clean_chess_moves.csv`.


## 1. Setup & Target Schema Definition
The target schema represents **one row per move (ply)**:

In [ ]:
import os
import glob
import json
import time
from collections import Counter
import chess
import chess.pgn
import pandas as pd
import numpy as np

PROCESSED_FILE = os.path.join("..", "data", "processed", "clean_chess_moves.csv")
AUDIT_FILE = os.path.join("..", "data", "processed", "intermediate", "cleaning_audit.json")
print("Clean dataset path:", os.path.abspath(PROCESSED_FILE))


## 2. Load Processed Clean Dataset & Cleaning Audit
Load the generated dataset and audit log.

In [ ]:
df = pd.read_csv(PROCESSED_FILE)
print(f"Loaded clean dataset: {len(df):,d} moves, {df.shape[1]} columns")

with open(AUDIT_FILE, "r", encoding="utf-8") as f:
    audit = json.load(f)

print("\n--- Cleaning Audit Removal Rules ---")
for rule in audit["removal_rules"]:
    print(f"[{rule['rule_id']}]: {rule['condition']}")
    print(f"  Action: {rule['action']} | Affected Count: {rule['affected_count']:,d}")
    print(f"  Reason: {rule['reason']}\n")


## 3. Dataset Validation & Integrity Checks
Verify missing values, chronological move order, and types.

In [ ]:
# Check missing values
missing = df.isnull().sum()
print("Missing values per column:")
for col, count in missing.items():
    pct = (count / len(df)) * 100
    print(f"  {col:20s}: {count:5d} null ({pct:.3f}%)")

# Verify chronological move ordering
grouped_plies = df.groupby("game_id")["ply"].apply(list)
is_ordered = all(p == list(range(1, len(p) + 1)) for p in grouped_plies)
print(f"\nChronological order within all {len(grouped_plies):,d} games verified: {is_ordered}")


## 4. Class Distribution & Key Behavioral Metrics
Inspect the verified Human vs AI move distribution, ratings, and thinking times.

In [ ]:
print("=== Label Distribution ===")
label_dist = df["label"].value_counts()
for lbl, cnt in label_dist.items():
    print(f"  {lbl:10s}: {cnt:7,d} ({cnt/len(df)*100:.2f}%)")

print("\n=== Game Phase Distribution ===")
phase_dist = df["game_phase"].value_counts()
for ph, cnt in phase_dist.items():
    print(f"  {ph:15s}: {cnt:7,d} ({cnt/len(df)*100:.2f}%)")

print("\n=== Move Time Summary (Seconds) ===")
print(df["move_time"].describe())

print("\n=== Player Rating Summary ===")
print(df["player_rating"].describe())


## 5. Sample Rows from the Processed Move Dataset
Inspect sample records across Opening, Middlegame, and Endgame.

In [ ]:
sample_cols = ["game_id", "player_id", "player_color", "move_number", "move", "move_time", "player_rating", "game_phase", "label"]
display(df[sample_cols].head(10))
